In [21]:
import pandas as pd
import json
from collections import defaultdict
import pickle
import os
import random
from tqdm import tqdm

In [22]:
quora_dev_qrels = pd.read_csv('quora/qrels/dev.tsv', sep='\t')
quora_qrels_dev_dict = defaultdict(list)
for index, row in quora_dev_qrels.iterrows():
    qid = str(row['query-id'])+"_q"
    quora_qrels_dev_dict[qid].append(str(row['corpus-id']))
len(quora_qrels_dev_dict.keys())

5000

In [23]:
quora_test_qrels = pd.read_csv('quora/qrels/test.tsv', sep='\t')
quora_qrels_test_dict = defaultdict(list)
for index, row in quora_test_qrels.iterrows():
    qid = str(row['query-id'])+"_q"
    quora_qrels_test_dict[qid].append(str(row['corpus-id']))
len(quora_qrels_test_dict.keys())

10000

In [24]:
with open('quora/queries.jsonl', 'r') as f:
	quora_queries = f.readlines()
quora_id_to_query_dict = {}
for line in quora_queries:
	data = json.loads(line)
	quora_id_to_query_dict[str(data['_id'])+"_q"] = data['text']
len(quora_id_to_query_dict.keys())

15000

In [25]:
with open('quora/corpus.jsonl', 'r') as f:
	quora_corpus = f.readlines()
quora_id_to_corpus_dict = {}
for line in quora_corpus:
	data = json.loads(line)
	if data["text"].strip() == "":
		continue
	quora_id_to_corpus_dict[str(data['_id'])] = data['text']
len(quora_id_to_corpus_dict.keys())

522931

In [26]:
trec_test_qrels = pd.read_csv('trec-covid/qrels/test.tsv', sep='\t')
trec_qrels_test_dict = defaultdict(list)
for index, row in trec_test_qrels.iterrows():
    qid = str(row['query-id'])+"_t"
    trec_qrels_test_dict[qid].append(row['corpus-id'])
len(trec_qrels_test_dict.keys())

50

In [27]:
with open('trec-covid/queries.jsonl', 'r') as f:
	trec_queries = f.readlines()
trec_id_to_query_dict = {}
for line in trec_queries:
	data = json.loads(line)
	trec_id_to_query_dict[str(data['_id'])+"_t"] = data['text']
len(trec_id_to_query_dict.keys())

50

In [28]:
with open('trec-covid/corpus.jsonl', 'r') as f:
	trec_corpus = f.readlines()
trec_id_to_corpus_dict = {}
for line in trec_corpus:
	data = json.loads(line)
	if data["text"].strip() == "":
		continue
	trec_id_to_corpus_dict[data['_id']] = data['text']
len(trec_id_to_corpus_dict.keys())

129192

In [29]:
os.makedirs('data', exist_ok=True)

In [30]:
train_qrel_dict = quora_qrels_test_dict.copy()
train_qrel_dict.update(dict(list(trec_qrels_test_dict.items())[8:]))
print("Train queries:", len(train_qrel_dict.keys()))

with open('data/train_qrels_dict.json', 'w') as f:
	json.dump(train_qrel_dict, f, indent=4)

Train queries: 10042


In [31]:
test_qrel_dict = quora_qrels_dev_dict.copy()
test_qrel_dict.update(dict(list(trec_qrels_test_dict.items())[:8]))
print("Test queries:", len(test_qrel_dict.keys()))

with open('data/test_qrels_dict.json', 'w') as f:
	json.dump(test_qrel_dict, f, indent=4)

Test queries: 5008


In [32]:
all_id_to_query_dict = {}
all_id_to_query_dict.update(quora_id_to_query_dict)
all_id_to_query_dict.update(trec_id_to_query_dict)
print("Total queries:", len(all_id_to_query_dict.keys()))

with open('data/id_to_query_dict.json', 'w') as f:
	json.dump(all_id_to_query_dict, f, indent=4)

Total queries: 15050


In [33]:
all_id_to_corpus_dict = {}
all_id_to_corpus_dict.update(quora_id_to_corpus_dict)
all_id_to_corpus_dict.update(trec_id_to_corpus_dict)
print("Total corpus documents:", len(all_id_to_corpus_dict.keys()))

with open('data/id_to_corpus_dict.json', 'w') as f:
	json.dump(all_id_to_corpus_dict, f, indent=4)

Total corpus documents: 652123


In [34]:
all_corpus_ids = all_id_to_corpus_dict.keys()

In [36]:
random.seed(42)

training_data = []
for qid in tqdm(train_qrel_dict.keys()):
	neg_doc_ids = list(all_corpus_ids - set(train_qrel_dict[qid]))
	for docid in train_qrel_dict[qid]:
		if docid not in all_id_to_corpus_dict:
			continue
		neg_docid = random.choice(neg_doc_ids)
		training_data.append((qid, docid, neg_docid))
len(training_data)

100%|██████████| 10042/10042 [09:02<00:00, 18.53it/s]


61377

In [37]:
with open('data/training_data.pkl', 'wb') as f:
	pickle.dump(training_data, f)